# Lesson 07 - Planning Design Pattern

Ipinapakita ng notebook na ito ang **Planning Design Pattern** para sa mga AI agent gamit ang Microsoft Agent Framework.
Matututuhan mo kung paano hatiin ang komplikadong mga kahilingan sa paglalakbay sa mga istrukturadong subtasks, italaga ang mga ito sa mga espesyalistang agent,
at isagawa ang nagresultang plano — lahat gamit ang istrukturadong output na pinapagana ng mga modelong Pydantic.


## Setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Pagkakawatak-watak ng Gawain

Ang pagkakawatak-watak ng gawain ang puso ng disenyo ng pattern sa pagpaplano. Sa halip na hilingin sa isang ahente na
hawakan ang isang kompleks na kahilingan mula simula hanggang katapusan, hinahati natin ang problema sa mas maliliit, malinaw na **mga subtasks**.
Ang bawat subtask ay itinalaga sa isang espesyalistang ahente (hal., mga flight, hotel, aktibidad) na may malinaw na
priyoridad at pagkakasunod-sunod ng mga dependency.

Ang paraang ito ay nagbibigay ng ilang mga benepisyo:
- **Kalinawan**: bawat subtask ay may isang tungkulin.
- **Pagsabay-sabay**: ang mga independent na subtasks ay maaaring sabay-sabay na tumakbo.
- **Katiyakan**: ang mga pagkabigo ay naiiwasan sa bawat indibidwal na subtask.
- **Pagsubaybay ng badyet**: ang mga gastos ay tinatantiya bawat subtask at pinagsama-sama.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Paggawa ng Planning Agent na may Nakaayos na Output

Ang planning agent ay gumaganap bilang **front desk coordinator**. Kapag binigyan ng mataas na antas ng kahilingan sa paglalakbay,
ito ay gumagawa ng nakaayos na `TravelPlan` — hinahati ang kahilingan sa mga subtask, nagtatakda ng mga prayoridad,
at tinutukoy ang mga dependencies upang ang concierge o execution layer ay makagawa ng gawain.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Pagsasagawa ng Plano gamit ang mga Espesyalistang Kasangkapan

Kapag ang front desk agent ay nakabuo na ng isang istrukturadong plano, ito ay isinasagawa ng **concierge agent**.
Bawat espesyalistang kasangkapan ay humahawak ng isang kategorya ng subtask (mga flight, hotel, mga gawain). Ang concierge
ay inuulit ang mga subtask ng plano ayon sa pagkakasunod-sunod ng dependency at ipinapadala ang bawat isa sa
angkop na kasangkapan.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Buod

Sa leksyong ito natutunan mo ang **Planning Design Pattern** para sa mga AI agents:

1. **Task Decomposition** — Ang isang front desk planning agent ay naghahati ng isang kumplikadong kahilingan sa paglalakbay sa
   mga nakabalangkas na subtasks gamit ang Pydantic models, nag-aassign ng bawat isa sa isang specialist agent na may mga prayoridad
   at mga dependencies.
2. **Structured Output** — Sa pamamagitan ng pagpasa ng `response_format` ang agent ay nagbabalik ng isang validated
   `TravelPlan` object sa halip ng free-form text, na nagpapasiguro ng maaasahang downstream processing.
3. **Plan Execution** — Ang concierge agent ay nag-iiterate sa subtasks gamit ang mga specialist tools
   (`book_flight`, `reserve_hotel`, `book_activity`) upang isagawa ang plano at iulat ang mga resulta.

Ang pattern na ito ay naghihiwalay ng *ano ang gagawin* (planning) mula sa *paano ito gagawin* (execution), na ginagawa ang mga agent
na mas modular, masusubok, at mas madaling palawakin.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
